# Salidas Estructuradas con Azure AI Assistants\nEste notebook demuestra cómo obtener una salida estructurada (JSON) utilizando la capacidad de llamada a funciones del SDK de Azure AI Assistants.

In [ ]:
# Importar librerías necesarias
import asyncio
import os
import json
from dotenv import load_dotenv
from azure.ai.assistant.aio import AssistantClient
from azure.ai.assistant.models import Assistant, AssistantThread, ToolDefinition, FunctionDefinition

## Cargar Variables de Entorno y Configurar Cliente

In [ ]:
load_dotenv()

client = AssistantClient(
    endpoint=os.environ.get("AZURE_AI_PROJECT_ENDPOINT"),
)

## Definir la Función y el Esquema de Salida

In [ ]:
# Definir la función que el asistente puede llamar
get_person_info_tool = ToolDefinition(
    function=FunctionDefinition(
        name="get_person_info",
        description="Extrae y registra la información de una persona.",
        parameters={
            "type": "object",
            "properties": {
                "name": {"type": "string", "description": "El nombre de la persona."},
                "age": {"type": "integer", "description": "La edad de la persona."},
                "occupation": {"type": "string", "description": "La ocupación de la persona."}
            },
            "required": ["name", "age", "occupation"]
        }
    )
)

## Crear el Asistente y el Hilo de Conversación

In [ ]:
async def create_assistant_and_thread():
    assistant = await client.create_assistant(
        model=os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME"),
        name="Asistente de Extracción de Datos",
        instructions="Eres un asistente que extrae información de texto no estructurado y la devuelve en formato JSON llamando a las funciones disponibles.",
        tools=[get_person_info_tool]
    )
    thread = await client.create_thread()
    return assistant, thread

assistant, thread = asyncio.run(create_assistant_and_thread())

## Obtener Salida Estructurada

In [ ]:
async def main():
    # Añadir mensaje al hilo
    await client.create_message(thread.id, content="Extrae los datos de John Doe, que tiene 42 años y es ingeniero de software.")

    # Ejecutar el asistente y esperar a que requiera una acción
    run = await client.create_run(assistant.id, thread.id)
    while run.status not in ['requires_action']:
        await asyncio.sleep(1)
        run = await client.get_run(thread.id, run.id)

    # Extraer los argumentos de la llamada a la función
    tool_call = run.required_action.submit_tool_outputs.tool_calls[0]
    function_args = json.loads(tool_call.function.arguments)

    print("Salida Estructurada Recibida:")
    print(json.dumps(function_args, indent=2))

    # Limpiar
    await client.delete_assistant(assistant.id)
    await client.delete_thread(thread.id)

if __name__ == "__main__":
    asyncio.run(main())